In [29]:
intents = {
  "menu": {
    "1": {"name": "pizza", "price": 199},
    "2": {"name": "burger", "price": 99},
    "3": {"name": "biryani", "price": 149},
    "4": {"name": "pasta", "price": 129},
    "5": {"name": "fries", "price": 79}
  },
  "intents": [
    {
      "tag": "greet",
      "patterns": ["hi", "hello", "hey"],
      "responses": ["Hello! What would you like to order?"]
    },
    {
      "tag": "ask_menu",
      "patterns": ["menu", "show menu", "food list"],
      "responses": ["Showing menu..."]
    },
    {
      "tag": "view_order",
      "patterns": ["my order", "what did i order", "show my order"],
      "responses": ["Here is your order."]
    },
    {
      "tag": "special_item",
      "patterns": ["special item", "what is special", "best item"],
      "responses": ["Our special items are Pizza and Biryani 😋"]
    },
    {
      "tag": "delivery_time",
      "patterns": ["when will order come", "delivery time", "when will it arrive"],
      "responses": ["Your order will arrive in 25–30 minutes 🚚"]
    },
    {
      "tag": "goodbye",
      "patterns": ["bye", "exit"],
      "responses": ["Goodbye!"]
    }
  ]
}

In [28]:
import random
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

texts = []
labels = []
responses = {}

for intent in intents["intents"]:
    tag = intent["tag"]
    responses[tag] = intent["responses"]

    for pattern in intent["patterns"]:
        texts.append(pattern.lower())
        labels.append(tag)

vectorizer = TfidfVectorizer(ngram_range=(1,2))
X = vectorizer.fit_transform(texts)

model = LogisticRegression()
model.fit(X, labels)

# ---------------------------
# STATE VARIABLES
# ---------------------------
cart = []
final_order = []
last_order = []
awaiting_selection = False
awaiting_confirmation = False

# ---------------------------
# HELPERS
# ---------------------------
def clean_text(text):
    text = text.lower()
    text = ''.join([c for c in text if c not in string.punctuation])

    # simple typo fixes
    text = text.replace("waht", "what")
    text = text.replace("willl", "will")

    return text

def predict_intent(text):
    X_test = vectorizer.transform([text])
    probs = model.predict_proba(X_test)
    confidence = max(probs[0])
    tag = model.classes_[probs.argmax()]
    if confidence < 0.4:
        return "fallback"
    return tag

def get_items_from_numbers(text):
    items = []
    for w in text.split():
        if w in intents["menu"]:
            items.append(intents["menu"][w]["name"])
    return items

def show_menu():
    print("Bot :")
    print("1. Pizza ₹199")
    print("2. Burger ₹99")
    print("3. Biryani ₹149")
    print("4. Pasta ₹129")
    print("5. Fries ₹79")
    print("Bot : Enter item number(s) (e.g., 1 or 1 2 3)")

# ---------------------------
# CHAT LOOP
# ---------------------------
print("Bot is running... (type 'exit' to stop)")

while True:
    user_input = input("You : ")
    cleaned = clean_text(user_input)

    if cleaned == "exit":
        print("Bot : Goodbye!")
        break

    # ---------------------------
    # MENU
    # ---------------------------
    if "menu" in cleaned:
        awaiting_selection = True
        show_menu()
        continue

    # ---------------------------
    # SELECT ITEMS
    # ---------------------------
    if awaiting_selection:
        items = get_items_from_numbers(cleaned)

        if items:
            cart = items
            awaiting_selection = False
            awaiting_confirmation = True
            print(f"Bot : You selected {', '.join(cart)}. Confirm?")
            continue
        else:
            print("Bot : Please select valid numbers (1-5).")
            continue

    # ---------------------------
    # CONFIRM ADD
    # ---------------------------
    if awaiting_confirmation:
        if cleaned in ["yes", "ok", "confirm", "haan", "sure"]:
            final_order.extend(cart)
            print(f"Bot : Added {', '.join(cart)} to your order.")
            cart = []
            awaiting_confirmation = False
            print("Bot : Do you want to add more? (yes/menu/checkout)")
            continue

        elif cleaned in ["no", "cancel"]:
            print("Bot : Okay 👍 You can checkout or add more items.")
            cart = []
            awaiting_confirmation = False
            continue

    # ---------------------------
    # ADD MORE
    # ---------------------------
    if cleaned in ["yes"] or "menu" in cleaned:
        if final_order:
            awaiting_selection = True
            show_menu()
            continue

    # ---------------------------
    # CHECKOUT
    # ---------------------------
    if "checkout" in cleaned or "done" in cleaned:
        if final_order:
            order_id = "ORD" + str(random.randint(1000, 9999))
            last_order = final_order.copy()

            print(f"Bot : Final Order: {', '.join(final_order)}")
            print(f"Bot : Order placed successfully! ID: {order_id}")

            final_order = []
        else:
            print("Bot : No items in your order.")
        continue

    # ---------------------------
    # VIEW ORDER
    # ---------------------------
    if "order" in cleaned or "booked" in cleaned:
        if final_order:
            print(f"Bot : Your current order: {', '.join(final_order)}")
        elif last_order:
            print(f"Bot : Your last order: {', '.join(last_order)}")
        else:
            print("Bot : No order found.")
        continue


    
    if "special" in cleaned:
        print("Bot : Our special items are Pizza and Biryani 😋")
        continue

    if "time" in cleaned or "when" in cleaned:
        print("Bot : Your order will arrive in 25–30 minutes 🚚")
        continue

    # ---------------------------
    # ML INTENT
    # ---------------------------
    intent = predict_intent(cleaned)

    if intent in responses:
        print("Bot :", random.choice(responses[intent]))
    else:
        print("Bot : Sorry, I didn't understand.")

Bot is running... (type 'exit' to stop)


KeyboardInterrupt: Interrupted by user

In [32]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("data.pkl", "wb") as f:
    pickle.dump(intents, f)

print("Saved successfully!")

Saved successfully!
